# Tier 1 Gatekeeper Training (ResNet-18)
**Project:** Gaze-Gated 5G Edge-to-Cloud AI  
**Description:** This notebook prepares a dataset of cropped lung nodules (Positive) and normal anatomy (Negative) from the LIDC-IDRI database. It trains a ResNet-18 Convolutional Neural Network to act as a "Gatekeeper" in our 5G MEC architecture, filtering out Edge AI false alarms before they trigger the heavy Vision-Language Model.

In [1]:
# Install required dependencies
!pip install -q pydicom pylidc tcia_utils opencv-python-headless scikit-learn

import os, glob, shutil, cv2, pydicom
import numpy as np
import pandas as pd

# --- PYTHON 3.13 & DEPENDENCY PATCHES ---
import configparser
if not hasattr(configparser, 'SafeConfigParser'):
    configparser.SafeConfigParser = configparser.ConfigParser
if not hasattr(np, 'int'):
    np.int = np.int64; np.float = np.float64; np.bool = np.bool_

with open(os.path.expanduser('~/.pylidcrc'), 'w') as f:
    f.write("[dicom]\npath = /content/raw_data\nwarn = True\n")

import pylidc as pl
from pylidc.utils import consensus
from tcia_utils import nbia

print("Environment Setup Complete. Ready for Data Extraction.")

/usr/local/lib/python3.13/dist-packages/pylidc/__init__.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as _pr


Environment Setup Complete. Ready for Data Extraction.


## Phase 1: Data Acquisition & Preprocessing
Downloads the top 50 highly-annotated patients from the TCIA database. Extracts 128x128 pixel crops of confirmed nodules (Label 1) and offset regions containing chest walls/blood vessels (Label 0) to train the model to reject false positives.

In [2]:
print("Fetching top 50 patients from TCIA...")
os.makedirs("/content/raw_data", exist_ok=True)
scans = pl.query(pl.Scan).all()

top_patients = []
for s in sorted(scans, key=lambda s: len(s.annotations), reverse=True):
    if s.patient_id not in top_patients: top_patients.append(s.patient_id)
    if len(top_patients) >= 50: break

df = pd.DataFrame(nbia.getSeries(collection="LIDC-IDRI"))
p_col = next((c for c in df.columns if 'patient' in c.lower() or 'subject' in c.lower()), None)
s_col = next((c for c in df.columns if 'uid' in c.lower() and 'series' in c.lower()), None)
m_col = next((c for c in df.columns if 'modality' in c.lower()), None)

subset = df[(df[m_col] == 'CT') & (df[p_col].isin(top_patients))].drop_duplicates(subset=p_col)
uids_to_download = subset[s_col].tolist()

if len(uids_to_download) > 0:
    nbia.downloadSeries(uids_to_download, input_type="list", path="/content/raw_data")

print("\nReorganizing DICOM folders...")
for series_dir in glob.glob("/content/raw_data/*"):
    if os.path.isdir(series_dir) and "LIDC" not in os.path.basename(series_dir):
        dcms = glob.glob(series_dir + "/*.dcm")
        if dcms:
            pid = pydicom.dcmread(dcms[0], stop_before_pixels=True).PatientID
            os.makedirs(os.path.join("/content/raw_data", pid), exist_ok=True)
            for dcm in dcms:
                try: shutil.move(dcm, os.path.join("/content/raw_data", pid))
                except: pass
        shutil.rmtree(series_dir)

print("\nExtracting Positive and Negative Crops...")
os.makedirs("/content/tier1_crops", exist_ok=True)
rows = []

for scan in pl.query(pl.Scan).filter(pl.Scan.patient_id.in_(top_patients)).all():
    try: vol = scan.to_volume()
    except: continue

    for idx, cluster in enumerate(scan.cluster_annotations()):
        try:
            if len(cluster) < 2: continue
            _, cbbox, _ = consensus(cluster, clevel=0.5)
            z = int((cbbox[2].start + cbbox[2].stop)/2)
            slice_hu = vol[:,:,z]

            # POSITIVE CROP (Real Nodule)
            y1, y2 = max(0, cbbox[0].start-16), min(slice_hu.shape[0], cbbox[0].stop+16)
            x1, x2 = max(0, cbbox[1].start-16), min(slice_hu.shape[1], cbbox[1].stop+16)
            if y1 >= y2 or x1 >= x2: continue

            # Apply Lung Windowing (-600 / 1500)
            pos_img = np.clip((slice_hu[y1:y2, x1:x2] - -600) / 1500 * 255 + 127.5, 0, 255).astype(np.uint8)
            cv2.imwrite(f"/content/tier1_crops/pos_{scan.patient_id}_{idx}.png", pos_img)
            rows.append({"path": f"/content/tier1_crops/pos_{scan.patient_id}_{idx}.png", "label": 1})

            # NEGATIVE CROP (Shifted out to normal anatomy)
            ny1, ny2 = max(0, y1-80), max(0, y1-80) + (y2-y1)
            nx1, nx2 = max(0, x1-80), max(0, x1-80) + (x2-x1)
            if ny1 >= ny2 or nx1 >= nx2: continue

            neg_img = np.clip((slice_hu[ny1:ny2, nx1:nx2] - -600) / 1500 * 255 + 127.5, 0, 255).astype(np.uint8)
            cv2.imwrite(f"/content/tier1_crops/neg_{scan.patient_id}_{idx}.png", neg_img)
            rows.append({"path": f"/content/tier1_crops/neg_{scan.patient_id}_{idx}.png", "label": 0})
        except Exception: pass

df_manifest = pd.DataFrame(rows)
df_manifest.to_csv("/content/tier1_manifest.csv", index=False)

if len(df_manifest) > 0: shutil.rmtree("/content/raw_data", ignore_errors=True)
print(f"\n✅ Data Prep Complete! Extracted {len(df_manifest)} training crops.")

Fetching top 50 patients from TCIA...

Reorganizing DICOM folders...

Extracting Positive and Negative Crops...
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... 

## Phase 2: ResNet-18 Model Training
Applies image augmentations (random flips and rotations) and trains a ResNet-18 CNN on the generated crops. The model uses Binary Cross Entropy Loss and an AdamW optimizer over 15 epochs.

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

class Tier1Dataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.RandomHorizontalFlip() if train else nn.Identity(),
            transforms.RandomRotation(15) if train else nn.Identity(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert("RGB")
        return self.transform(img), torch.tensor(row['label'], dtype=torch.float32)

# Load data and split
df = pd.read_csv("/content/tier1_manifest.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
train_loader = DataLoader(Tier1Dataset(train_df, True), batch_size=16, shuffle=True)
val_loader = DataLoader(Tier1Dataset(val_df, False), batch_size=16, shuffle=False)

# Setup ResNet-18
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 1) # Binary output
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
loss_fn = nn.BCEWithLogitsLoss()
best_auc = 0

print("Starting Tier 1 Training...\n")
for epoch in range(1, 16):
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(imgs.to(device)), labels.to(device).unsqueeze(1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy().flatten()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy().flatten())

    try: auc = roc_auc_score(all_labels, all_probs)
    except: auc = 0
    acc = np.mean((np.array(all_probs) > 0.5) == np.array(all_labels))

    print(f"Epoch {epoch:02d}/15 | Loss: {train_loss/len(train_loader):.4f} | Val AUC: {auc:.4f} | Val Acc: {acc:.4f}")
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), "/content/tier1_best.pth")

print(f"\n🏆 Training Complete! Best Val AUC: {best_auc:.4f}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]


Starting Tier 1 Training...

Epoch 01/15 | Loss: 0.3278 | Val AUC: 0.9865 | Val Acc: 0.9231
Epoch 02/15 | Loss: 0.1241 | Val AUC: 0.9906 | Val Acc: 0.9103


KeyboardInterrupt: 

# Download


In [ ]:
from google.colab import files
import os

if os.path.exists('/content/tier1_best.pth'):
    print("Triggering download for Tier 1 Gatekeeper Model...")
    files.download('/content/tier1_best.pth')
else:
    print("Error: Model file not found.")